In [5]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV

data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
Y = data.target

In [6]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)
rf = RandomForestClassifier(random_state=42)

In [7]:
random_parameter_space = {
    'n_estimators': np.arange(10, 500, 50),
    'max_depth': [None, 10, 20, 30, 40, 50],
    'min_samples_split': [2, 5, 10],
    'bootstrap': [True, False]
}

rs = RandomizedSearchCV(
    estimator=rf, 
    param_distributions=random_parameter_space, 
    n_iter=10,      
    cv=5,           # 5-fold Cross-Validation
    verbose=1,      # value of verbose equal to 1 enables printing of basic updates while model building
    n_jobs=-1       # Hyperparameter to use all cores of our CPU
)

rs.fit(X_train, Y_train)
print(f"Best parameters from Random Search: {rs.best_params_}")

accuracy1 = rs.score(X_test, Y_test)
print(f"Accuracy after using Grid Search with Cross-Validation: {accuracy1:.4f}")

Fitting 5 folds for each of 10 candidates, totalling 50 fits
Best parameters from Random Search: {'n_estimators': np.int64(260), 'min_samples_split': 5, 'max_depth': 40, 'bootstrap': False}
Accuracy after using Grid Search with Cross-Validation: 0.9649


In [8]:
grid_param_space = {
    'n_estimators': [80, 100, 120],
    'max_depth': [rs.best_params_['max_depth']], #We are using what we found the best using Random Search
    'min_samples_split': [2, 3]
}

gs = GridSearchCV(
    estimator=rf, 
    param_grid=grid_param_space, 
    cv=5,           
    verbose=1, 
    n_jobs=-1
)

gs.fit(X_train, Y_train)
print(f"\nFinal Best Parameters: {gs.best_params_}")

accuracy2 = gs.score(X_test, Y_test)
print(f"Accuracy on unseen test data: {accuracy2:.4f}")

Fitting 5 folds for each of 6 candidates, totalling 30 fits

Final Best Parameters: {'max_depth': 40, 'min_samples_split': 3, 'n_estimators': 100}
Accuracy on unseen test data: 0.9649
